In [2]:
from datasets import load_dataset

import os
HF_TOKEN = os.getenv("HF_TOKEN", "YOUR_HF_TOKEN_HERE")

dataset = load_dataset('MichalMlodawski/closed-open-eyes', token=HF_TOKEN)
# dataset = load_dataset('dataset_name', split='train')

Generating train split: 100%|██████████| 126560/126560 [02:11<00:00, 958.88 examples/s] 


In [3]:
!nvidia-smi

Sun Oct  4 21:39:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.36                 Driver Version: 566.36         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   42C    P8              3W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from datasets import DatasetDict, Dataset
import numpy as np

full_train_dataset = dataset['train']

split_dataset = full_train_dataset.train_test_split(test_size=0.2, seed=42) 

train_dataset = split_dataset['train']
validation_dataset = split_dataset['test']

validation_test_split = validation_dataset.train_test_split(test_size=0.5, seed=42)  

validation_dataset = validation_test_split['train']
test_dataset = validation_test_split['test']

dataset_split = DatasetDict({
    'train': train_dataset,
    'val': validation_dataset,
    'test': test_dataset
})

print("Training size:", len(dataset_split['train']))
print("Validation size:", len(dataset_split['val']))
print("Test size:", len(dataset_split['test']))


Training size: 101248
Validation size: 12656
Test size: 12656


In [5]:
dataset_split['val'].select(range(500))

Dataset({
    features: ['Image_id', 'Left_eye_react', 'Right_eye_react', 'Label', 'Image_data'],
    num_rows: 500
})

In [6]:
dataset_split['train']

Dataset({
    features: ['Image_id', 'Left_eye_react', 'Right_eye_react', 'Label', 'Image_data'],
    num_rows: 101248
})

In [7]:
import os

import cv2

def coco_to_yolo(x, y, w, h, width, height):
    return [((2*x + w)/(2*width)), ((2*y + h)/(2*height)), w/width, h/height]

def preprocessing(partition: str, data: object, max_samples: int = 5000):
    os.makedirs(f"datasets/images/{partition}", exist_ok=True)
    os.makedirs(f"datasets/labels/{partition}", exist_ok=True)

    num_samples = min(max_samples, len(data[partition]))
    samples = data[partition].select(range(num_samples))
    
    for i, sample in enumerate(samples):
        if sample['Label'] == 'open_eyes':
            label_val = 0
            text_image = "open"
        elif sample['Label'] == 'closed_eyes':
            label_val = 1
            text_image = "close"
        else:
            continue

        labels = [label_val, label_val]
        img = sample['Image_data']['file']
        bboxes = [sample['Left_eye_react'], sample['Right_eye_react']]
        width = int(img.size[0])
        height = int(img.size[1])

        items = []
        for label, box in zip(labels, bboxes):
            xc, yc, w, h = coco_to_yolo(box[0], box[1], box[2], box[3], width, height)
            items.append(f"{label} {xc} {yc} {w} {h}")

        with open(f"datasets/labels/{partition}/{i}_{text_image}.txt", "w") as f:
            for item in items:
                f.write(f"{item}\n")

        img.save(f"datasets/images/{partition}/{i}_{text_image}.png")


In [8]:

preprocessing("train", dataset_split)
preprocessing("val", dataset_split)
preprocessing("test", dataset_split)